In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS housekg.etl.realty_price_fact (
  slug STRING,
  price LONG,
  effective_from TIMESTAMP,
  effective_to TIMESTAMP,
  is_current BOOLEAN
)

In [ ]:
from pyspark.sql import functions as F, types as T

from awsglue.context import GlueContext
from pyspark.context import SparkContext

glueContext = GlueContext(SparkContext())
spark = glueContext.spark_session
df_bronze  = spark.read.format("delta") \
    .load("s3://housekg-etl-bucket/delta/bronze/")
cleaned_df_bronze = df_bronze.withColumn(
    "kitchen_square",
    F.when(F.col("kitchen_square").isNull(), F.when(F.col("square") > 120, 15).otherwise(
    F.when(F.col("square") < 70, 6).otherwise(F.round(F.col("square") * 0.15).cast("int")))).otherwise(F.col("kitchen_square"))
).withColumn(
    "ceiling_height",
    F.when(F.col('ceiling_height').isNull(), 3).otherwise(F.col('ceiling_height'))
).withColumn('toilet', F.when(F.col('toilet').isNull(), F.when(F.col('square') > 140, F.lit(3)).otherwise(F.when(F.col('square') > 750, F.lit(2)).otherwise(F.lit(1)))).otherwise(F.col('toilet'))
             ).withColumn('updated_at',
        F.unix_timestamp(
            'updated_at',
            'yyyy-MM-dd HH:mm:ss'
        )
        ).dropDuplicates(["slug"]).alias('incoming')

In [ ]:
scd2_table = spark.table('housekg.etl.realty_price_fact')
current_scd2_df = scd2_table.where(col('is_current') == True).alias('current')
join_cond = [
    current_scd2_df["slug"] == cleaned_df_bronze["slug"]
]
joined_df = cleaned_df_bronze.join(current_scd2_df, join_cond, how="left_outer")

# Detect new or changed prices
changed_df = joined_df.filter(
    (current_scd2_df["price"].isNull()) |  # new entry
    (cleaned_df_bronze["price"] != current_scd2_df["price"])  # price changed
).select(cleaned_df_bronze["slug"], cleaned_df_bronze["price"])

# Close old records
updates = changed_df.join(
    current_scd2_df, "slug", "inner"
).withColumn("is_current", F.lit(False)) \
 .withColumn("effective_to", F.current_timestamp()) \
 .select(
    "slug", "incoming.price", "effective_from", "effective_to", "is_current"
)

# Insert new records
new_records = changed_df.withColumn("effective_from", F.current_timestamp()) \
    .withColumn("effective_to", F.lit(None).cast("timestamp")) \
    .withColumn("is_current", F.lit(True))

# Apply updates: close old records
if updates.count() > 0:
    updates.createOrReplaceTempView("updates_temp")
    spark.sql(f"""
      MERGE INTO housekg.etl.realty_price_fact AS target
      USING updates_temp AS source
      ON target.slug = source.slug
      AND target.is_current = true
      WHEN MATCHED THEN
        UPDATE SET
          target.effective_to = source.effective_to,
          target.is_current = false
    """)
    print("✅ Closed old records.")

# Insert new records
if new_records.count() > 0:
    new_records.write.mode("append").saveAsTable('housekg.etl.realty_price_fact')
    print("✅ Inserted new records.")

cleaned_df_bronze.drop('price').write.mode("overwrite").saveAsTable('housekg.etl.realty_object_dim')